# Tutorial 04 — Turbo Equalization Loop

The receiver runs the BCJR equalizer and the BCJR convolutional decoder alternately, exchanging extrinsic LLRs via the interleaver. Each iteration halves the error tail until the EXIT tunnel closes. This tutorial measures per-iteration BER for a single Eb/N0 point.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from nsm.modem.msprs import precompute as nsm_pre, coded_ber
from nsm.codec.conv import precompute as conv_pre
from nsm.interleaver import random_indices
from nsm.channel.awgn import setup

In [ ]:
L0, SOURCE_BITS, MAX_ITERS = 3, 1000, 7
CODER = {'K': 3, 'octal_code': (0o5, 0o7)}
code   = conv_pre(CODER, SOURCE_BITS)
nsm    = nsm_pre(L0, code['coding_length'], 'unbalanced')
idx    = random_indices(code['coding_length'], seed=42)
ch     = setup((3, 3, 1), 0.5, rate=0.5); nv, ns = ch['noise_var'][0], ch['noise_std'][0]

## One packet, accumulate errors per iteration

In [ ]:
errs_total = np.zeros(MAX_ITERS + 1, dtype=np.int64); pkts = 30
for _ in range(pkts):
    e = coded_ber(SOURCE_BITS, code['coding_length'], code['polynomials'],
                   code['n_outputs'], code['memory'], code['total_states'],
                   code['next_states'], code['outputs'],
                   nsm['modulation_length'], L0, nsm['h0'], nsm['h1'],
                   nsm['branch_labels'], nsm['memory'], nsm['total_states'],
                   nsm['next_states'], nsm['branch_indices'],
                   idx, MAX_ITERS, nv, ns)
    errs_total += np.asarray(e, dtype=np.int64)
ber = errs_total / (pkts * SOURCE_BITS)
print('per-iter BER:', np.round(ber, 5))

## Plot

In [ ]:
plt.semilogy(range(MAX_ITERS + 1), np.clip(ber, 1e-6, 1), 'o-')
plt.xlabel('turbo iteration'); plt.ylabel('BER'); plt.grid(True, which='both', alpha=0.3)
plt.title(f'Per-iteration BER at Eb/N0 = 3 dB, L0={L0}, unbalanced')